In [1]:
import sys
!{sys.executable} -m pip install azure-ai-ml azure-identity --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
adlfs 2025.8.0 requires fsspec>=2023.12.0, but you have fsspec 2023.10.0 which is incompatible.
azure-cli 2.81.0 requires azure-datalake-store~=1.0.1, but you have azure-datalake-store 0.0.53 which is incompatible.
azure-cli 2.81.0 requires azure-keyvault-keys==4.11.0, but you have azure-keyvault-keys 4.8.0 which is incompatible.
azure-cli 2.81.0 requires azure-mgmt-keyvault==12.1.0, but you have azure-mgmt-keyvault 10.3.1 which is incompatible.
azure-cli 2.81.0 requires azure-mgmt-storage==24.0.0, but you have azure-mgmt-storage 22.0.0 which is incompatible.
azure-cli 2.81.0 requires websocket-client~=1.3.1, but you have websocket-client 1.9.0 which is incompatible.
azureml-automl-dnn-nlp 1.61.0 requires torch==2.2.2, but you have torch 2.9.1 which is incompatible.
azureml-automl-runtime 1.61.0 requires psutil<5.

In [1]:
from azure.ai.ml import MLClient, command, Input
from azure.ai.ml.entities import Environment
from azure.identity import DefaultAzureCredential
from pathlib import Path

In [2]:
ml_client = MLClient.from_config(credential=DefaultAzureCredential())

print(f"Conectado a workspace: {ml_client.workspace_name}")
print(f"Resource group: {ml_client.resource_group_name}")

Found the config file in: /config.json
Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Conectado a workspace: mlw-churn-dev
Resource group: rg-churn-dev


In [8]:
env = Environment(
    name="churn-sklearn13",
    description="Sklearn 1.3 for class_weight support",
    conda_file={
        "channels": ["conda-forge"],
        "dependencies": [
            "python=3.10",
            "scikit-learn=1.3.0",
            "pandas",
            "numpy",
            "joblib",
        ]
    },
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04",
)

env = ml_client.environments.create_or_update(env)
print(f"✅ Environment: {env.name}:{env.version}")

✅ Environment: churn-sklearn13:1


In [4]:
# Determinar project root
current_dir = Path.cwd()

if current_dir.name == 'notebooks':
    project_root = current_dir.parent
else:
    project_root = current_dir

print(project_root)

/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-churn-dev/code/Users/asandovh/telco-churn


In [9]:
# Crear job
training_job  = command(
    code="../",
    command="python src/train.py --learning-rate 0.05 --max-depth 3",
    environment="churn-sklearn13:1",
    compute="cpu-cluster",
    experiment_name="churn-prediction-training",
    display_name="churn-sklearn13:1",
    description="Churn prediction with class_weight=balanced",
)

submitted_job = ml_client.jobs.create_or_update(training_job)

print(f"\nJob enviado!")
print(f"Nombre: {submitted_job.name}")
print(f"Status: {submitted_job.status}")
print(f"Studio URL: {submitted_job.studio_url}")
print(f"\nVer job en Azure ML Studio:")
print(f"   {submitted_job.studio_url}")

Uploading telco-churn (2.68 MBs): 100%|██████████| 2678551/2678551 [00:00<00:00, 4224389.09it/s]





Job enviado!
Nombre: mighty_turtle_xm8m6lb3s2
Status: Starting
Studio URL: https://ml.azure.com/runs/mighty_turtle_xm8m6lb3s2?wsid=/subscriptions/9cbbe496-fe29-459a-a3d7-44790ebac1fb/resourcegroups/rg-churn-dev/workspaces/mlw-churn-dev&tid=3048dc87-43f0-4100-9acb-ae1971c79395

Ver job en Azure ML Studio:
   https://ml.azure.com/runs/mighty_turtle_xm8m6lb3s2?wsid=/subscriptions/9cbbe496-fe29-459a-a3d7-44790ebac1fb/resourcegroups/rg-churn-dev/workspaces/mlw-churn-dev&tid=3048dc87-43f0-4100-9acb-ae1971c79395
